### Bag of Stories
**Description:** Compute various correlations between surprise and predictability measures for a set of stories.

**Contributor:** Florentina Armaselu  

In [90]:
# Import required packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from pathlib import Path
import os
import datetime as dt
import seaborn as sns
import re

Define the input, output paths.

In [105]:
# Define the base directory relative to the current script location
base_dir = os.getcwd()
# Set the path to the input folder for the KLD results on the human-written stories
input_kld_hum = os.path.join(base_dir, "./../results/human/kld/grimm/csv/")
# Set the path to the output folder for KLD results on the AI-generated stories with equal-length start
input_kld_ai_gen_el_st = os.path.join(base_dir, "./../results/ai-gen/kld/grimm/el_start/csv/")
# Set the path of the input folder for KLD results on the AI-generated stories with text-tiling start
input_kld_ai_gen_tt_st = os.path.join(base_dir, "./../results/ai-gen/kld/grimm/tt_start/csv/")
# Set the path to the input folder for the SGS results on the human-written stories
input_sgs_hum = os.path.join(base_dir, "./../results/human/seg-sim/grimm/csv/")
# Set the path to the input folder for SGS results on the AI-generated stories with equal-length start
input_sgs_ai_gen_el_st = os.path.join(base_dir, "./../results/ai-gen/seg-sim/grimm/el_start/csv/")
# Set the path to the input folder for SGS results on the AI-generated stories with text-tiling start
input_sgs_ai_gen_tt_st = os.path.join(base_dir, "./../results/ai-gen/seg-sim/grimm/tt_start/csv/")
# Set the path to the input folder for the JSD results on the human-written stories
input_jsd_hum = os.path.join(base_dir, "./../results/human/jsd/grimm/csv/")
# Set the path to the input folder for JSD results on the AI-generated stories with equal-length start
input_jsd_ai_gen_el_st = os.path.join(base_dir, "./../results/ai-gen/jsd/grimm/el_start/csv/")
# Set the path to the input folder for JSD results on the AI-generated stories with text-tiling start
input_jsd_ai_gen_tt_st = os.path.join(base_dir, "./../results/ai-gen/jsd/grimm/tt_start/csv/")
# Set the path to the input folder for sentiment analysis results
input_sent = os.path.join(base_dir, "./../results/sentiment_analysis/grimm/")
# Set the path to the output folder for the summary tables
output_summ = os.path.join(base_dir, "./../results/summaries/grimm/")
# Set the path to the output folder for the correlation results
output = os.path.join(base_dir, "./../results/correlations/grimm/")

Define constants.

In [104]:
# String constants for comparison types
COMP_ADJ = 'adj'  # Label for adjacent comparison
COMP_CTX = 'ctx'  # Label for contextual comparison

# String constants for story types
STORY_HUM = 'hum'  # Label for human stories
STORY_GEN = 'gen'  # Label for AI-generated stories
STORY_POP = 'pop' # Label for popular stories
STORY_FORG = 'forg' # Label for forgotten stories

# String constants for start types of AI-generated stories
GEN_EL_ST = 'el_st' # Label for AI-generated stories with equal length start
GEN_TT_ST = 'tt_st' # Label for AI-generated stories with text-tiling start

# String constants for segmentation methods
SEG_EL = 'el'  # Label for equal-length segmentation
SEG_TT = 'tt'  # Label for text-tiling segmentation

# String constants for output subfolders and file formats
OUT_CSV = 'csv'
OUT_PNG = 'png'

# String constant for metric labels
KLD = 'kld' # Label for Kullback-Leibler divergence
SGS = 'sgs' # Label for segment similarity
JSD = 'jsd' # Label for Jensen-Shannon divergence
SENT = 'snt' # Label for sentiment analysis

# String constant for calculated measures
AVG = 'Avg' # Label for average per story
DEV = 'Dev' # Label for deviation per story
AVG_RUN = 'Avg Run' # Label for average per run
STD_RUN = 'Std Run' # Label for standard deviation per run
VADER = 'Vader' # Label for Vader sentiment analysis values
TXT_BLOB = 'TextBlob' # Label for TextBlob sentiment analysis values 
SUBJ = 'Subjectivity' # Label for Subjectivity sentiment analysis values

# String constant for the summary and correlation file name prefix
SUMM_FN_PREF = 'summary'
CORR_FN_PREF = 'correlation'

Define the function to build the result summary tables used in computing correlations.

In [ ]:
def build_result_summaries(start_type, comp_meth, seg_type, metrics, story_type):
    """
    Build the summary tables used in computing correlations.
    Arguments: start_type (str): Type of the start of AI-generated stories.
               comp_meth (str): Comparison method to be used.
               seg_type (str): Segmentation type to be used.
               metrics (list): List of metrics for which the result summaries are computed.
    """
    summ_string_init = f"{comp_meth}_{seg_type}_"
    if start_type:
        summ_string_init = f"{start_type}_{summ_string_init}"

    # Initialize an empty DataFrame for the summary
    # summ_df = pd.DataFrame(columns=['ID'])
    summ_df = pd.DataFrame()

    timestamp = dt.datetime.now().strftime('%Y%m%d_%H%M%S')  # Create a timestamp for the output files

    # Set the input paths for human stories
    for i, metric in enumerate(metrics):
        summ_string = summ_string_init
        if metric == SENT:
            input = input_sent
        if story_type == STORY_HUM:
            if metric == KLD:
                input = input_kld_hum
            elif metric == SGS:
                input = input_sgs_hum
            elif metric == JSD:
                input = input_jsd_hum
            summ_string += f"{metric}_{STORY_HUM}"
        elif story_type == STORY_GEN:
            if start_type == GEN_EL_ST:
                if metric == KLD:
                    input = input_kld_ai_gen_el_st
                elif metric == SGS:
                    input = input_sgs_ai_gen_el_st
                elif metric == JSD:
                    input = input_jsd_ai_gen_el_st
            elif start_type == GEN_TT_ST:
                if metric == KLD:
                    input = input_kld_ai_gen_tt_st
                elif metric == SGS:
                    input = input_sgs_ai_gen_tt_st
                elif metric == JSD:
                    input = input_jsd_ai_gen_tt_st
            summ_string += f"{metric}_{STORY_GEN}"
        
        col_avg = f"{metric.upper()} {AVG}"
        col_dev = f"{metric.upper()} {DEV}"
        col_avg_run = f"{metric.upper()} {AVG_RUN}"
        col_std_run = f"{metric.upper()} {STD_RUN}"
        col_avg_vader = f"{AVG}_{VADER}"
        col_avg_txt_blob = f"{AVG}_{TXT_BLOB}"
        col_avg_subj = f"{AVG}_{SUBJ}"

        # Read the summary files for the specified metric
        for file_path in Path(input).glob("*.csv"):
            if not file_path.name.startswith(f"{SUMM_FN_PREF}_{summ_string}"):
                continue
            else:
                with open(file_path, "r", encoding="utf-8") as input_file:
                    df = pd.read_csv(input_file)
                    if i == 0:
                        summ_df = df
                    else: 
                        if metric == SENT:
                            sel_cols = [col_avg_vader, col_avg_txt_blob, col_avg_subj] # Select the columns to keep in the summary DataFrame
                        else:
                            sel_cols = [col_avg, col_dev, col_avg_run, col_std_run] # Select the columns to keep in the summary DataFrame
                        summ_df[sel_cols] = df[sel_cols].values
        
        if i == len(metrics)-1:
            summ_string = summ_string.replace(f"{metric}_", "")
        
    # Save the summary DataFrame to a CSV file
    output_file = os.path.join(output_summ, f"{SUMM_FN_PREF}_{summ_string}_{timestamp}.{OUT_CSV}")
    summ_df.to_csv(output_file, index=False)

    print(f"Summary file saved to: {Path(output_file).name}")

Define the functions for computing correlations and visualise them.

In [176]:
# Create plot and save csv and png files for the correlations
def plot_and_save(corr_matrix, fn_suff):
    """
        Create the plot for a correlation matrix.
        Save the correlation matrix and the plot as csv and png files.
    """
      # Create a timestamp for the output files
    timestamp = dt.datetime.now().strftime('%Y%m%d_%H%M%S')

    # Plot the correlations
    plt.figure(figsize=(9.5, 7.5))
    sns.heatmap(corr_matrix, cmap="YlGnBu", vmin = -1, vmax = 1, center = 0, annot=True, square=True)
    plt.title(f"Grimm folktales: {fn_suff} heatmap")
    plt.tight_layout(rect=[0, 0, 1, 0.98])

    # Save the csv and png files
    csv_path = os.path.join(output, OUT_CSV, f"{CORR_FN_PREF}_{fn_suff}_{timestamp}.{OUT_CSV}")
    corr_matrix.to_csv(csv_path, index=False)
    png_path = os.path.join(output, OUT_PNG, f"{CORR_FN_PREF}_{fn_suff}_{timestamp}.{OUT_PNG}")
    plt.savefig(png_path, bbox_inches='tight')
    plt.close()

# Compute correlations for all summary files
def compute_correlations():
    """
        Read the summary files and compute correlations.
    """
    input = output_summ
    df = pd.DataFrame()
    cnt_proc = 0

    # Read the summary files for the specified metric
    for file_path in Path(input).glob("*.csv"):
        if not file_path.name.startswith(SUMM_FN_PREF):
            continue
        else:
            with open(file_path, "r", encoding="utf-8") as input_file:
                fn_suff = file_path.name.replace(f"{SUMM_FN_PREF}_", "")
                idx = re.search(r"_\d", fn_suff).start() # Extract the file name suffix
                fn_suff = fn_suff[:idx]  # Slice the string up to the index
                df = pd.read_csv(input_file)
                cdf = df.drop(df.filter(regex='ID|File|Story|Run').columns, axis=1)
                cdf_pop = cdf.loc[0:9] # Select only the rows for the popular tales
                cdf_forg = cdf.loc[10:19] # Select only the rows for the forgotten tales
                
                # Compute correlation matrix and plot correlation heatmaps
                corr_matrix = cdf.corr(method='pearson', min_periods=1)
                corr_matrix_pop = cdf_pop.corr(method='pearson', min_periods=1)
                corr_matrix_forg = cdf_forg.corr(method='pearson', min_periods=1)

                plot_and_save(corr_matrix, fn_suff)
                plot_and_save(corr_matrix_pop, f"{STORY_POP}_{fn_suff}")
                plot_and_save(corr_matrix_forg, f"{STORY_FORG}_{fn_suff}")

                cnt_proc += 3
    
    print(f"End processing: {cnt_proc} files saved to the {Path(output).name} folder.")


Validate the input and build the summaries.  

In [ ]:
# Build the result summary files
build_summ = False # Set to True to build the summary tables used in computing the correlations (run only once), False otherwise
start_types = {GEN_EL_ST, GEN_TT_ST}  # Choose between GEN_EL_START or GEN_TT_START for AI-generated stories or None for human stories
comp_meths = {COMP_ADJ, COMP_CTX}  # Choose between COMP_ADJ or COMP_CTX for adjacent or contextual comparison
seg_types = {SEG_EL, SEG_TT}  # Choose between SEG_EL or SEG_TT for equal-length or text-tiling segmentation
metrics = sorted({KLD, SGS, JSD, SENT})  # List of metrics: KLD, SGS, JSD, SENT
story_types = {STORY_HUM, STORY_GEN}  # Choose between STORY_HUM or STORY_GEN for human or AI-generated stories

if build_summ:
    # Validate the segmentation method and set the input path accordingly
    if not seg_types or not all(seg_type in [SEG_EL, SEG_TT] for seg_type in seg_types):
        raise ValueError("Invalid segmentation method. Choose from ", SEG_EL, "or ", SEG_TT, ".")
    
    # Validate the comparison method
    if not comp_meths or not all(comp_meth in [COMP_ADJ, COMP_CTX] for comp_meth in comp_meths):
        raise ValueError("Invalid comparison method. Choose from ", COMP_ADJ, "or ", COMP_CTX, ".")
    
    # Validate the metrics
    if not metrics or not all(metric in [KLD, SGS, JSD, SENT] for metric in metrics):
        raise ValueError("Invalid metrics. Choose from ", KLD, ", ", SGS, ", ", JSD, ", ", SENT, ".")
    
    # Validate the story type
    if not story_types or not all(story_type in [STORY_HUM, STORY_GEN] for story_type in story_types):
        raise ValueError("Invalid story type. Choose from ", STORY_HUM, "or ", STORY_GEN, ".")
    
    for story_type in story_types:
        for comp_meth in comp_meths:
            for seg_type in seg_types:
                for start_type in start_types:
                    if story_type == STORY_HUM:
                        start_type = None
                    # Build the summary tables used in computing correlations
                    build_result_summaries(start_type, comp_meth, seg_type, metrics, story_type) 




Validate the input and compute the correlations.

In [ ]:
# Compute correlations
compute_correlations()
